# Database Inventory Project

In [1]:
first_barcode_list = ['049000028904', '051000000114', '013000006031', '044000032029', '037000511612', '035000510800', '038000387537', '037600106253']
second_barcode_list = ['013000000031', '04252614', '5012345678900', '40123455', '10049000028901', '0-684-84328-5', '978-0-684-84328-5', '2049-3630', '9772049363004']

## Barcode Detection & Validation 

There are many types of barcodes: UPC-A, UPC-E, EAN-8, EAN-13, GTIN-14, ITF-14, ISBN-10, ISBN-13, ISSN, CODE39, CODE128, CODABAR, and MSI are all options that are available. 
A fundamental part of the inventory process is being able to collect a set of items to be barcoded. The objective is for a user to scan any type of barcode and for that object to be classified correct on the first attempt. 

### Detection:


### Validation Class Structure:

Every barcode type needs to be validated against the checksum conditions. Check sum condtions are a set of rules that generate the barcode, to show that it's feasible and to distinguish it from other codes. 

In [2]:
from abc import ABC, abstractmethod

class BarcodeValidator(ABC):
    name = ""
    valid_lengths = ()

    def matches_length(self, code):
        return len(code) in self.valid_lengths
    
    @abstractmethod
    def check_sum(self, code):
        pass

    def validate(self, code):
        if not self.matches_length(code):
            return False
        return self.check_sum(code)


In [ ]:
class UPCValidator(BarcodeValidator):
    name = "UPC-A"
    valid_lengths = (12,)

    def check_sum(self, code):
        odd_sum = sum(int(code[i]) for i in range(0, len(code), 2))
        even_sum = sum(int(code[i]) for i in range(1, len(code), 2))
        total_sum = odd_sum * 3 + even_sum
        return total_sum % 10 == 0
    
class EAN13Validator(BarcodeValidator):
    name = "EAN-13"
    valid_lengths = (13,)

    def check_sum(self, code):
        if code.startswith("978") or code.startswith("979"):
            return False
        else:
            odd_sum = sum(int(code[i]) for i in range(0, len(code), 2))
            even_sum = sum(int(code[i]) for i in range(1, len(code), 2))
            total_sum = odd_sum + even_sum * 3
            return total_sum % 10 == 0
    
class EAN8Validator(BarcodeValidator):
    name = "EAN-8"
    valid_lengths = (8,)

    def check_sum(self, code):
        odd_sum = sum(int(code[i]) for i in range(0, len(code), 2))
        even_sum = sum(int(code[i]) for i in range(1, len(code), 2))
        total_sum = odd_sum * 3 + even_sum
        return total_sum % 10 == 0
    
class GTIN14Validator(BarcodeValidator):
    name = "GTIN"
    valid_lengths = (14,)

    def check_sum(self, code):
        odd_sum = sum(int(code[i]) for i in range(0, len(code), 2))
        even_sum = sum(int(code[i]) for i in range(1, len(code), 2))
        total_sum = odd_sum * 3 + even_sum
        return total_sum % 10 == 0
    
class ISBN10Validator(BarcodeValidator):
    name = "ISBN-10"
    valid_lengths = (10,)

    def check_sum(self, code):
        total = sum((10 - i) * int(code[i]) for i in range(9))
        total += int(code[-1])
        return total % 11 == 0
    
class ISBN13Validator(BarcodeValidator):
    name = "ISBN-13"
    valid_lengths = (13,)

    def check_sum(self, code):
            if code.startswith("978") or code.startswith("979"):
                odd_sum = sum(int(code[i]) for i in range(0, len(code), 2))
                even_sum = sum(int(code[i]) for i in range(1, len(code), 2))
                total_sum = odd_sum + even_sum * 3
                return total_sum % 10 == 0
            else:
                return False
    
class ISSNValidator(BarcodeValidator):
    name = "ISSN"
    valid_lengths = (8,)

    def check_sum(self, code):
        total_sum = sum((8 - i) * int(code[i]) for i in range(len(code) - 1))
        check_digit = int(code[-1])
        return (total_sum + check_digit) % 11 == 0

validators = [
    UPCValidator(),
    EAN13Validator(),
    EAN8Validator(),
    GTIN14Validator(),
    ISBN10Validator(),
    ISBN13Validator(),
    ISSNValidator()
]

In [29]:
for val in validators:
    print(f"{val.name} valid lengths: {val.valid_lengths}")

UPC-A valid lengths: (12,)
EAN-13 valid lengths: (13,)
EAN-8 valid lengths: (8,)
GTIN valid lengths: (14,)
ISBN-10 valid lengths: (10,)
ISBN-13 valid lengths: (13,)
ISSN valid lengths: (8,)


In [37]:
class BarcodeDetector:

    def __init__(self, validators):
        self.validators = validators

    def detect(self, code):
        code = code.replace("-", "").replace(" ", "")
        for validator in self.validators:
            if validator.matches_length(code) and validator.validate(code):
                return validator.name
    

detector = BarcodeDetector(validators)
for code in second_barcode_list:
    print(f"Code: {code}, Detected Types: {detector.detect(code)}")



Code: 013000000031, Detected Types: UPC-A
Code: 04252614, Detected Types: None
Code: 5012345678900, Detected Types: EAN-13
Code: 40123455, Detected Types: EAN-8
Code: 10049000028901, Detected Types: GTIN
Code: 0-684-84328-5, Detected Types: ISBN-10
Code: 978-0-684-84328-5, Detected Types: EAN-13
Code: 2049-3630, Detected Types: ISSN
Code: 9772049363004, Detected Types: None
